# Word Embeddings: One-Hot Encoding vs. Dense Semantic Vectors (Word2Vec)

**Environment:** Google Colab / Local Jupyter Notebook (Python 3.9+)  
**Domain:** Cloud Computing, Systems Engineering & Modern Software Architecture

---

### What You Will Learn
1. **The Representation Problem:** Why machines require numeric vector representations of discrete language.
2. **One-Hot Encoding from Scratch:** Constructing sparse vocabulary vectors and measuring memory scaling ($O(|V|)$).
3. **The Orthogonality Flaw:** Mathematical proof of why one-hot vectors cannot represent semantic similarity ($S_C(u, v) = 0$).
4. **Dense Continuous Embeddings:** The Distributional Hypothesis ("You shall know a word by the company it keeps").
5. **Word2Vec Architectures:** Continuous Bag of Words (CBOW) vs. Continuous Skip-Gram.
6. **Vector Geometry & Semantic Algebra:** Calculating vector offsets, finding analogies, and measuring nearest neighbors.
7. **2D Latent Space Visualizations:** Using PCA and t-SNE to discover semantic clustering.
8. **Hyperparameter Tuning:** Evaluating context window sizes and embedding dimensionality.

In [ ]:
# Install required dependencies
!pip install -q gensim scikit-learn matplotlib seaborn pandas numpy

## 1. The Classical Approach: One-Hot Encoding

In traditional NLP, text is treated as a set of discrete symbols. In **One-Hot Encoding**, every distinct word in a vocabulary $V$ is assigned an index $i$. A word vector has length $|V|$, where index $i$ is set to `1` and all remaining positions are set to `0`.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder

# Technical vocabulary from systems and cloud architecture
tech_vocab = [
    "server", "container", "cluster", "database", "cloud",
    "compiler", "interpreter", "runtime", "kernel", "memory"
]

# 1. Manual One-Hot Vector generation
vocab_to_id = {word: idx for idx, word in enumerate(tech_vocab)}
id_to_vocab = {idx: word for word, idx in vocab_to_id.items()}
V_size = len(tech_vocab)

def word_to_one_hot(word, vocab_map):
    vec = np.zeros(len(vocab_map), dtype=int)
    if word in vocab_map:
        vec[vocab_map[word]] = 1
    return vec

print("Vocabulary Mapping:")
for word, idx in vocab_to_id.items():
    print(f"  {word:<12} -> Index {idx}")

print("\nSample One-Hot Vector for 'cluster':")
print(word_to_one_hot("cluster", vocab_to_id))

### 1.1 The Orthogonality & Semantic Blindness Flaw

Let us calculate the **Cosine Similarity** between every pair of one-hot encoded words.
Cosine similarity is given by:
$$
	ext{Cosine Similarity}(u, v) = rac{u \cdot v}{\|u\|_2 \|v\|_2}
$$

Because each one-hot vector has a single `1` at a unique index, the dot product of two distinct one-hot vectors is always `0`:
$$
u \cdot v = 0 \quad (orall u 
eq v)
$$

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Build one-hot matrix for all words in vocabulary
one_hot_matrix = np.array([word_to_one_hot(w, vocab_to_id) for w in tech_vocab])

# Compute pairwise cosine similarity
sim_matrix = cosine_similarity(one_hot_matrix)
df_one_hot_sim = pd.DataFrame(sim_matrix, index=tech_vocab, columns=tech_vocab)

print("One-Hot Pairwise Cosine Similarity Matrix (First 5x5):")
display(df_one_hot_sim.iloc[:5, :5])

### 1.2 Limitations of One-Hot Vectors:
1. **Complete Orthogonality:** `"container"` is just as mathematically distant from `"cluster"` as it is from `"kernel"`. There is zero semantic proximity.
2. **Sparsity & Memory Explosion:** If a dictionary contains 100,000 words, each word requires a vector of 100,000 dimensions—99.999% of which are useless zeros.
3. **Out of Vocabulary (OOV) Fragility:** New words cannot be represented without resizing the entire vector space and retraining all downstream models.

---

## 2. Dense Representations: Distributed Semantics & Word2Vec

The **Distributional Hypothesis** (J.R. Firth, 1957) states:
> *"You shall know a word by the company it keeps."*

Instead of sparse binary vectors, **dense word embeddings** represent words as real-valued continuous vectors in a lower-dimensional space (e.g., 25 to 300 dimensions). Words that occur in similar linguistic contexts are mapped to nearby coordinates in vector space.

### Word2Vec Architectures:
1. **Continuous Bag of Words (CBOW):** Predicts the target center word given its surrounding context words. Fast, well-suited for frequent words.
2. **Continuous Skip-Gram:** Predicts surrounding context words given the center target word. Better for small datasets and rare words.

In [ ]:
import random
from gensim.models import Word2Vec

# Domain-specific technical training corpus
technical_corpus = [
    # Cluster & Containerization contexts
    ["docker", "runs", "container", "on", "linux", "server"],
    ["kubernetes", "manages", "container", "cluster", "on", "cloud"],
    ["container", "runs", "microservices", "inside", "cluster"],
    ["docker", "builds", "image", "for", "microservices"],
    ["kubernetes", "schedules", "pod", "on", "worker", "node"],
    ["cluster", "scales", "worker", "node", "in", "cloud"],
    
    # Systems & Operating System contexts
    ["linux", "kernel", "manages", "memory", "and", "cpu"],
    ["operating", "system", "kernel", "allocates", "memory", "buffer"],
    ["cpu", "executes", "instructions", "from", "memory"],
    ["process", "threads", "run", "on", "cpu", "cores"],
    ["linux", "server", "monitors", "cpu", "memory", "usage"],
    
    # Storage & Database contexts
    ["database", "stores", "records", "in", "storage", "cluster"],
    ["postgresql", "database", "manages", "relational", "records"],
    ["redis", "database", "caches", "records", "in", "memory"],
    ["cloud", "database", "provides", "scalable", "storage"],
    
    # Programming & Compilation contexts
    ["compiler", "translates", "source", "code", "to", "binary"],
    ["rust", "compiler", "guarantees", "memory", "safety"],
    ["python", "interpreter", "executes", "script", "code"],
    ["developer", "writes", "source", "code", "for", "microservices"],
    ["engineer", "deploys", "container", "to", "cloud", "server"]
]

print(f"Total training sentences: {len(technical_corpus)}")
print(f"Total vocabulary size: {len(set(w for s in technical_corpus for w in s))} distinct tokens")

### 2.1 Training CBOW vs. Skip-Gram Models

We train two Word2Vec models on our technical corpus:
- `sg=0` -> Continuous Bag-of-Words (CBOW)
- `sg=1` -> Continuous Skip-Gram

In [ ]:
# Train Continuous Bag of Words (CBOW)
model_cbow = Word2Vec(
    sentences=technical_corpus,
    vector_size=32,      # Dense dimensionality
    window=3,           # Context window distance
    min_count=1,        # Include rare words for this small corpus
    sg=0,               # CBOW
    epochs=100,         # Training iterations
    workers=1,
    seed=42             # Deterministic reproducibility
)

# Train Continuous Skip-Gram
model_skipgram = Word2Vec(
    sentences=technical_corpus,
    vector_size=32,
    window=3,
    min_count=1,
    sg=1,               # Skip-Gram
    epochs=100,
    workers=1,
    seed=42
)

print("Dense vector representation for 'kubernetes' (CBOW, 32-d):")
print(np.round(model_cbow.wv["kubernetes"], 3))

## 3. Semantic Similarity in Dense Vector Space

Now let us test if our trained dense vectors understand technical relationships. Unlike One-Hot vectors, similar concepts should yield high cosine similarities!

In [ ]:
test_words = ["container", "kernel", "database", "compiler"]

print("=== CBOW Most Similar Terms ===")
for target in test_words:
    neighbors = model_cbow.wv.most_similar(target, topn=3)
    formatted = ", ".join([f"{word} ({score:.2f})" for word, score in neighbors])
    print(f"  {target:<10} -> {formatted}")

print("\n=== Skip-Gram Most Similar Terms ===")
for target in test_words:
    neighbors = model_skipgram.wv.most_similar(target, topn=3)
    formatted = ", ".join([f"{word} ({score:.2f})" for word, score in neighbors])
    print(f"  {target:<10} -> {formatted}")

### 3.1 Pairwise Semantic Comparison

Let us compare word pairs from the same technical subfield vs. across different subfields.

In [ ]:
evaluation_pairs = [
    # Within same conceptual domain (expected HIGH)
    ("docker", "kubernetes"),
    ("kernel", "cpu"),
    ("database", "records"),
    ("compiler", "code"),
    
    # Across different domains (expected LOWER)
    ("kubernetes", "kernel"),
    ("docker", "compiler"),
    ("database", "cpu"),
    ("redis", "compiler")
]

results = []
for w1, w2 in evaluation_pairs:
    sim_cbow = model_cbow.wv.similarity(w1, w2)
    sim_sg = model_skipgram.wv.similarity(w1, w2)
    results.append({
        "Word 1": w1,
        "Word 2": w2,
        "CBOW Similarity": round(float(sim_cbow), 4),
        "Skip-Gram Similarity": round(float(sim_sg), 4)
    })

pd.DataFrame(results)

### 3.2 Finding the Concept Outlier (`doesnt_match`)
Dense embeddings can identify semantic anomalies—given a group of related words and one unrelated word, which one does not belong?

In [ ]:
groups = [
    ["docker", "kubernetes", "container", "kernel"],   # kernel is OS-level, others are containerization
    ["cpu", "memory", "kernel", "microservices"],       # microservices is software architecture
    ["compiler", "interpreter", "code", "cloud"]       # cloud is infrastructure
]

for grp in groups:
    outlier = model_skipgram.wv.doesnt_match(grp)
    print(f"Group: {grp}")
    print(f"  -> Outlier detected: '{outlier}'\n")

## 4. Visualizing Embeddings in 2D Space

To visually verify that semantically related technical concepts cluster together, we project the 32-dimensional embeddings into 2D space using **Principal Component Analysis (PCA)**.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# Select curated domain vocabulary
selected_vocab = [
    # Containers & Cloud
    "docker", "kubernetes", "container", "cluster", "cloud",
    # OS & Hardware
    "linux", "kernel", "cpu", "memory",
    # Data & Storage
    "database", "redis", "storage", "records",
    # Code & Compilation
    "compiler", "code", "binary", "developer"
]

vectors = np.array([model_skipgram.wv[w] for w in selected_vocab])

# Reduce from 32D to 2D
pca = PCA(n_components=2, random_state=42)
coords_2d = pca.fit_transform(vectors)

plt.figure(figsize=(10, 7))
plt.scatter(coords_2d[:, 0], coords_2d[:, 1], color="#2563eb", edgecolors="black", s=100, alpha=0.8)

for i, word in enumerate(selected_vocab):
    plt.annotate(
        word,
        xy=(coords_2d[i, 0], coords_2d[i, 1]),
        xytext=(6, 4),
        textcoords="offset points",
        fontsize=11,
        weight="bold"
    )

plt.title("2D PCA Projection of Technical Word Embeddings (Word2Vec)", fontsize=14, weight="bold")
plt.xlabel(f"PC 1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)")
plt.ylabel(f"PC 2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)")
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()

## 5. Hyperparameter Exploration: Effect of Window Size

The `window` parameter specifies how many tokens to the left and right of the target word are treated as context.
- **Small window ($w=1$):** Captures immediate syntactic and functional role.
- **Large window ($w=4$ or $5$):** Captures broad topical and domain relatedness.

In [ ]:
for win in [1, 2, 4]:
    exp_model = Word2Vec(
        sentences=technical_corpus,
        vector_size=32,
        window=win,
        min_count=1,
        sg=1,
        epochs=80,
        workers=1,
        seed=42
    )
    sim = exp_model.wv.similarity("docker", "kubernetes")
    print(f"Context Window = {win} -> Similarity('docker', 'kubernetes') = {sim:.4f}")

## 6. Practice Exercises (Solved)

### Exercise 1: Pairwise Heatmap Matrix
Generate a heatmap comparing the similarity of technical keywords across categories.

In [ ]:
import seaborn as sns

eval_terms = ["docker", "container", "cluster", "kernel", "cpu", "database", "compiler"]
eval_matrix = np.zeros((len(eval_terms), len(eval_terms)))

for i, w1 in enumerate(eval_terms):
    for j, w2 in enumerate(eval_terms):
        eval_matrix[i, j] = model_skipgram.wv.similarity(w1, w2)

plt.figure(figsize=(8, 6))
sns.heatmap(eval_matrix, annot=True, fmt=".2f", xticklabels=eval_terms, yticklabels=eval_terms, cmap="Blues")
plt.title("Word2Vec Pairwise Cosine Similarity Heatmap")
plt.show()

### Exercise 2: Vector Arithmetic & Analogy
Test if vector subtraction and addition capture relationships:
$$\vec{v}_{\text{kubernetes}} - \vec{v}_{\text{docker}} + \vec{v}_{\text{compiler}} \approx ?$$

In [ ]:
# positive=[A, B], negative=[C] calculates: A + B - C
analogy_result = model_skipgram.wv.most_similar(
    positive=["kubernetes", "compiler"],
    negative=["docker"],
    topn=2
)
print("Analogy (kubernetes - docker + compiler):")
for word, score in analogy_result:
    print(f"  {word}: {score:.4f}")

## Key Takeaways

| Metric / Dimension | One-Hot Encoding | Word2Vec (Dense Embeddings) |
|---|---|---|
| **Dimensionality** | Sparse, $|V| = 50,000+$ | Dense, typically $50 - 300$ |
| **Semantic Meaning** | None (All dot products are $0$) | High (Proximity indicates shared context) |
| **Memory Scaling** | Quadratic in vocabulary size | Compact, fixed-size matrix |
| **New Words** | Requires expanding dimension | Requires subword models (FastText/BPE) |
| **Primary Use** | Categorical feature encoding | Modern NLP, Search, RAG retrieval |